In [2]:
# numpy and pandas for data manipulation
import numpy as np
import pandas as pd 
from sklearn.model_selection import train_test_split

# File system manangement
import os

# Suppress warnings 
import warnings
warnings.filterwarnings('ignore')

# matplotlib and seaborn for plotting
import matplotlib.pyplot as plt
import seaborn as sns




from sklearnex import patch_sklearn
patch_sklearn()  # patches scikit-learn algorithms
# from sklearnex import unpatch_sklearn
# unpatch_sklearn()


# ------------------- IMPORT SRC ------------------------------------
# src is the parent folder of notebooks, so we need to add it to sys.path to import config and utils
import sys
notebook_dir = os.getcwd() 

# Parent folder of src
project_root = os.path.abspath(os.path.join(notebook_dir, "..")) 
sys.path.append(project_root)

print("sys.path contains:", sys.path[-1])

from src import config, utils  
# -------------------------------------------------------

sys.path contains: /home/ismail/Documents/projects/ml-projects/icr


Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [4]:
# -------------------------------
# Load Data
# -------------------------------
X_train = pd.read_csv('../data/X_train.csv')
X_test  = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv')
# test_ids = X_test['PassengerId']
# X_test.drop(columns=['PassengerId'], inplace=True)

# Flatten target if needed
# Map target to numeric
target = config.Target

y_train_numeric = y_train[target]


num_classes = len(np.unique(y_train_numeric))
print(f"Number of classes: {num_classes}")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Number of classes: 2
X_train shape: (617, 56)
X_test shape: (5, 57)


In [11]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
# y_test_enc  = le.transform(y_test)

In [7]:

cat_features = X_train.select_dtypes(
    include=['object', 'category', 'bool']
).columns.tolist()

print("Categorical features:", cat_features)


Categorical features: ['EJ']


In [8]:



# Replace NaN with a string and ensure string dtype
for col in cat_features:
    X_train[col] = X_train[col].fillna('__MISSING__').astype(str)
    X_test[col]  = X_test[col].fillna('__MISSING__').astype(str)


cat_idx = [X_train.columns.get_loc(col) for col in cat_features]

# Convert cat_features to pd.Categorical dtype
for col in cat_features:
    X_train[col] = pd.Categorical(X_train[col])
    X_test[col] = pd.Categorical(X_test[col])

In [ ]:
import optuna
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score
import numpy as np
import time
import gc


# -------------------------------
# Optuna objective for CatBoost
# -------------------------------
def objective(trial):

    params = {
        'loss_function': 'Logloss',
        'eval_metric': 'Logloss',
        # 'task_type': 'GPU',          # GPU training
        'devices': '0',
        # bootstrap_type: ['Bayesian', 'No','Bernoulli'] seems important feature, check it => Each produces slightly different trees → useful for ensembling later.
        'random_seed': 42,
        'verbose': 0,
        'auto_class_weights':'Balanced',

        # Core boosting params
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 100.0, log=True),

        # Sampling / regularization
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 0.0, 10.0),
        'border_count': trial.suggest_int('border_count', 32, 255),

        # Overfitting control
        'od_type': 'Iter',
        'od_wait': 100,
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    logloss_scores = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, y_train_enc)):

        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_val = y_train_enc[train_idx], y_train_enc[valid_idx]

        train_pool = Pool(
            data=X_tr,
            label=y_tr,
            cat_features=cat_idx
        )

        valid_pool = Pool(
            data=X_val,
            label=y_val,
            cat_features=cat_idx
        )

        model = CatBoostClassifier(
            **params,
            iterations=5000
        )

        model.fit(
            train_pool,
            eval_set=valid_pool,
            use_best_model=True,
            verbose=False
        )

        pred_val = model.predict_proba(valid_pool)
        fold_logloss = log_loss(y_val, pred_val)
        logloss_scores.append(fold_logloss)

        trial.report(fold_logloss, step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(logloss_scores)


In [13]:
# -------------------------------
# Optuna study
# -------------------------------
debug = True
timeout = 60 if debug else 3600

start = time.time()

study = optuna.create_study(
    direction='minimize',
    pruner=optuna.pruners.SuccessiveHalvingPruner(
        min_resource=2,
        reduction_factor=4,
        min_early_stopping_rate=1
    )
)

study.optimize(objective, n_trials=30)

end = time.time()

print(f"Optuna finished in {end - start:.2f} seconds")
print("Best params:", study.best_params)
print("Best CV Binary logloss:", study.best_value)


[I 2026-03-03 00:23:47,799] A new study created in memory with name: no-name-d800b800-2eff-4e92-a818-9f5352814eb5
[I 2026-03-03 00:23:53,796] Trial 0 finished with value: 0.2513494369004673 and parameters: {'learning_rate': 0.1814442105879416, 'depth': 8, 'l2_leaf_reg': 0.05041903947614276, 'bagging_temperature': 0.786671281991384, 'random_strength': 0.18854512335209295, 'border_count': 194}. Best is trial 0 with value: 0.2513494369004673.
[I 2026-03-03 00:23:56,896] Trial 1 finished with value: 0.2639712332164832 and parameters: {'learning_rate': 0.057906406697541754, 'depth': 7, 'l2_leaf_reg': 0.003527593875441008, 'bagging_temperature': 0.772125773785782, 'random_strength': 2.8844433967668803, 'border_count': 219}. Best is trial 0 with value: 0.2513494369004673.
[I 2026-03-03 00:26:53,298] Trial 2 finished with value: 0.19701724403576046 and parameters: {'learning_rate': 0.18005147783368258, 'depth': 10, 'l2_leaf_reg': 29.206071015030002, 'bagging_temperature': 0.8078190475210518, '

Optuna finished in 440.05 seconds
Best params: {'learning_rate': 0.023251622206339667, 'depth': 6, 'l2_leaf_reg': 48.133409089103054, 'bagging_temperature': 0.9840524497786919, 'random_strength': 3.437194637244614, 'border_count': 121}
Best CV Binary logloss: 0.1642802576774684


In [10]:
# -------------------------------
# Train final CatBoost model
# -------------------------------
best_params = study.best_params
best_params.update({
    'loss_function': 'Logloss',
    'eval_metric': 'Logloss',
    'task_type': 'GPU',
    'devices': '0',
    'random_seed': 2026,
    'verbose': 100
})

train_pool_full = Pool(
    X_train,
    label=y_train_enc,
    cat_features=cat_idx
)

test_pool_full = Pool(
    X_test,
    label=y_test_enc,
    cat_features=cat_idx
)

print("\nTraining final CatBoost model...")
final_model = CatBoostClassifier(
    **best_params,
    iterations=5000
)

final_model.fit(
    train_pool_full,
    eval_set=test_pool_full,
    use_best_model=True
)

# -------------------------------
# Evaluate on test set
# -------------------------------
pred_test = final_model.predict_proba(test_pool_full)  # returns 2D: [:,1] is prob of class 1
pred_test_prob = pred_test[:, 1]  # probability of positive class
pred_test_class = (pred_test_prob >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test_enc, pred_test_prob)
logloss = log_loss(y_test_enc, pred_test_prob)
accuracy = accuracy_score(y_test_enc, pred_test_class)

print("\n--- Test set performance ---")
print(f"ROC AUC (OVR weighted): {roc_auc:.5f}")
print(f"Log Loss:              {logloss:.5f}")
print(f"Accuracy:              {accuracy:.5f}")

gc.collect()



Training final CatBoost model...
0:	learn: 0.6913668	test: 0.6914066	best: 0.6914066 (0)	total: 88.3ms	remaining: 7m 21s
100:	learn: 0.6258686	test: 0.6302763	best: 0.6302763 (100)	total: 20.3s	remaining: 16m 26s
200:	learn: 0.6102268	test: 0.6195398	best: 0.6195398 (200)	total: 46.4s	remaining: 18m 28s
300:	learn: 0.6019830	test: 0.6158872	best: 0.6158872 (300)	total: 1m 14s	remaining: 19m 17s
400:	learn: 0.5960276	test: 0.6141498	best: 0.6141498 (400)	total: 1m 43s	remaining: 19m 52s
500:	learn: 0.5911011	test: 0.6132161	best: 0.6131999 (497)	total: 2m 11s	remaining: 19m 44s
600:	learn: 0.5869933	test: 0.6124607	best: 0.6124607 (591)	total: 2m 38s	remaining: 19m 23s
700:	learn: 0.5830490	test: 0.6121902	best: 0.6121874 (695)	total: 3m 11s	remaining: 19m 32s
800:	learn: 0.5795003	test: 0.6119281	best: 0.6119281 (800)	total: 3m 45s	remaining: 19m 42s
900:	learn: 0.5755920	test: 0.6116324	best: 0.6115960 (884)	total: 4m 17s	remaining: 19m 29s
1000:	learn: 0.5717822	test: 0.6114653	best

0